<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/13-performance-tuning/04-cluster-sizing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 13.4 — Cluster sizing: the math, since local mode can't show the cluster

Local-mode caveat (12.4's, again): one JVM plays every executor, so
this notebook can't demonstrate a real multi-executor cluster or
dynamic allocation scaling up/down. What it CAN do: build the sizing
CALCULATOR you'd actually use, print every dynamic-allocation config
and what it means, and connect cores/memory back to 12.1's model with
real numbers.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("13.4-cluster-sizing")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## The sizing calculator: cluster spec -> executor shape

Implements the "5 cores, ~4x GB" rule of thumb as an actual function,
not just prose — given a cluster's machine spec, propose a shape.

In [ ]:
def propose_executor_shape(cores_per_machine, memory_gb_per_machine,
                            num_machines, cores_per_executor=5):
    """13.4's rule of thumb, as code: cap cores/executor around 5,
    leave headroom for the OS/YARN daemons, and size memory
    proportionally (~4x cores_per_executor GB, minus overhead)."""
    usable_cores = cores_per_machine - 1          # leave 1 core for OS/daemons
    executors_per_machine = max(1, usable_cores // cores_per_executor)
    mem_per_executor_gb = memory_gb_per_machine // executors_per_machine
    total_executors = executors_per_machine * num_machines

    print(f"machines: {num_machines} x ({cores_per_machine} cores, {memory_gb_per_machine}GB)")
    print(f"proposed: --executor-cores {cores_per_executor}")
    print(f"proposed: --executor-memory {mem_per_executor_gb}g "
          f"(remember: 12.4's memoryOverhead is ADDITIONAL, ~10% on top)")
    print(f"proposed: --num-executors {total_executors} "
          f"({executors_per_machine} per machine)")
    print(f"total task parallelism: {total_executors * cores_per_executor} "
          f"concurrent tasks across the cluster")

propose_executor_shape(cores_per_machine=16, memory_gb_per_machine=128, num_machines=10)

## What local[*] actually gives you today

Confirm the local session's real parallelism, for contrast with the
calculator's cluster-scale numbers above.

In [ ]:
print("local[*] cores (defaultParallelism):", spark.sparkContext.defaultParallelism)
print("this is the ENTIRE 'cluster' for this notebook -- 1 JVM, 1 executor-equivalent")

## Dynamic allocation configs, printed and explained inline

None of these do anything meaningful in `local[*]` (no real executors
to add/remove), but confirm you can read and set them, and know what
each governs.

In [ ]:
dynamic_alloc_configs = {
    "spark.dynamicAllocation.enabled":          "master switch",
    "spark.dynamicAllocation.minExecutors":     "floor -- never scale below this",
    "spark.dynamicAllocation.maxExecutors":     "ceiling -- never scale above this",
    "spark.dynamicAllocation.initialExecutors": "starting point before any scaling signal",
    "spark.shuffle.service.enabled":            "REQUIRED with dynamic allocation -- "
                                                 "decouples shuffle files from executor lifetime",
}
for k, note in dynamic_alloc_configs.items():
    try:
        v = spark.conf.get(k)
    except Exception:
        v = "<unset>"
    print(f"{k:<45} = {v:<8}  # {note}")

## Why the shuffle service matters: reproduce the DEPENDENCY, not the danger

We can't safely demonstrate an executor being killed mid-shuffle
locally. Instead, confirm shuffle files persist independent of any
one task's lifetime -- the property the shuffle service generalizes
to executor lifetime on a real cluster (10.2's shuffle files, revisited).

In [ ]:
df = spark.range(500_000).withColumn("k", (col("id") % 20).cast("int"))
df.groupBy("k").count().collect()   # writes real shuffle files to spark.local.dir

import glob
shuffle_files = glob.glob(os.path.join(
    spark.conf.get("spark.local.dir", "/tmp"), "**", "shuffle_*"), recursive=True)
print(f"shuffle files still on disk after the job: {len(shuffle_files)} "
      f"(they outlive the TASK that wrote them -- the shuffle service "
      f"generalizes this to outlive the EXECUTOR too)")

In [ ]:
spark.stop()

## Exercises

1. Use `propose_executor_shape` with `cores_per_executor=10` instead
   of 5. What does the total parallelism number do, and connect the
   trade-off back to the lesson's two reasons for capping at ~5.
2. Extend the calculator to also print `memoryOverhead` explicitly
   (12.4's `max(384m, 0.10 x executor-memory)` default) as a separate
   line from `executor-memory`.
3. For a workload that's 90% light (4-way parallelism) and 10% heavy
   (40-way parallelism), compute the WASTED executor-hours under
   static allocation at `num-executors=40` for the whole job's
   duration, versus what dynamic allocation between `min=4,max=40`
   would use. Make reasonable time assumptions and show the math.
4. Given the calculator's output for a 10-machine cluster, how would
   the shape change if half the machines only had 64GB instead of
   128GB? Would you size for the smaller machines or run two
   different shapes?
5. Explain in your own words why dynamic allocation absolutely
   requires the external shuffle service, using what you saw about
   shuffle files persisting past task completion in the last cell.

In [ ]:
# your exercise code here